# Aula 12: Dominando Parâmetros de Inferência de LLMs e Avaliação Contínua com DeepEval


## 1. O Arsenal de Parâmetros

*   **Temperature (Temperatura):** Controla a aleatoriedade. Valores próximos a `0.0` tornam o modelo determinístico (pega sempre o token mais provável). Valores maiores (ex: `0.8 - 1.0`) aumentam a criatividade, achatando a distribuição de probabilidades.
*   **Max New Tokens:** Limita estritamente o tamanho da resposta gerada, essencial para controle de custos e latência.
*   **Repetition Penalty (Penalidade de Repetição):** Penaliza tokens que já apareceram no texto. Ótimo para modelos que tendem a entrar em loops de repetição.


In [29]:
# Instalação das bibliotecas necessárias
!pip install -q openai deepeval nest-asyncio

In [30]:
import os
from google.colab import userdata
import nest_asyncio
from openai import OpenAI

# DeepEval utiliza async por baixo dos panos
nest_asyncio.apply()

# Configure suas chaves de API nos Secrets do Colab
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Inicializando o cliente da OpenAI
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

## 2. Experimento Prático: Criatividade vs Determinismo

Vamos testar duas configurações diferentes para o mesmo prompt e observar as respostas.

In [46]:
prompt = "Explique o conceito de gravidade quântica em apenas um parágrafo."

# Configuração 1: Analítica e Determinística (Ideal para RAG e Extração de Dados)
params_deterministic = {
    "temperature": 0.1,
    "max_tokens": 150
}

# Configuração 2: Criativa e Solta (Ideal para Brainstorming e Copywriting)
params_creative = {
    "temperature": 1.5,
    "presence_penalty": 0.6,
    "max_tokens": 150
}

def generate_response(prompt, params):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        **params
    )
    return response.choices[0].message.content

resp_det = generate_response(prompt, params_deterministic)
resp_cre = generate_response(prompt, params_creative)

print("=== Resposta Determinística ===\n", resp_det, "\n")
print("=== Resposta Criativa ===\n", resp_cre, "\n")

=== Resposta Determinística ===
 A gravidade quântica é uma área da física teórica que busca unificar a mecânica quântica, que descreve as interações das partículas subatômicas, com a relatividade geral, que explica a gravidade como a curvatura do espaço-tempo causada pela presença de massa. O desafio central da gravidade quântica é entender como a gravidade se comporta em escalas extremamente pequenas, onde os efeitos quânticos se tornam significativos, e como isso se relaciona com a estrutura do espaço-tempo. Modelos como a teoria das cordas e a gravidade quântica em loop são tentativas de formular uma teoria coerente que possa descrever a gravidade em termos quânticos 

=== Resposta Criativa ===
 A gravidade quântica é uma tentativa de unificar a mecânica quântica, que descreve as interações das partículas em escala subatômica, com a teoria da relatividade geral de Einstein, que explica a gravidade como a curvarão de espaço-tempo por massa e energia. Esse campo de pesquisa busca ent

## 3. Avaliação Científica com DeepEval

Olhar no olho ("eyeballing") não escala em produção. Vamos usar o **DeepEval** para medir o quão engajadora (criativa) e quão concisa (focada) são as respostas. O DeepEval usa métricas baseadas em LLMs para avaliar critérios complexos.

In [48]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate

# Definindo uma métrica customizada para avaliar "Criatividade e Engajamento"
creativity_metric = GEval(
    name="Criatividade",
    criteria="você deve avaliar se um estudante de graduaão de fisica que está no porimeiro ano possa entender. Determine se a explicação utiliza analogias criativas, vocabulário rico e é engajadora para o leitor.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.5
)

# Casos de teste
tc_det = LLMTestCase(input=prompt, actual_output=resp_det)
tc_cre = LLMTestCase(input=prompt, actual_output=resp_cre)

# Avaliando
print("Avaliando Resposta Determinística:")
creativity_metric.measure(tc_det)
print(f"Score: {creativity_metric.score}, Reason: {creativity_metric.reason}\n")

print("Avaliando Resposta Criativa:")
creativity_metric.measure(tc_cre)
print(f"Score: {creativity_metric.score}, Reason: {creativity_metric.reason}")


Output()

Avaliando Resposta Determinística:


Output()

Score: 0.640733340004593, Reason: A explicação é clara e compreensível para um estudante de graduação do primeiro ano, utilizando vocabulário acessível e explicando os conceitos básicos de gravidade quântica, mecânica quântica e relatividade geral. No entanto, o texto não faz uso de analogias criativas para facilitar o entendimento, o que limita o engajamento e o interesse do leitor. O texto é informativo, mas falta um elemento mais envolvente ou ilustrativo que poderia tornar o conceito mais acessível e interessante.

Avaliando Resposta Criativa:


Score: 0.6399811640739795, Reason: A explicação é compreensível para um estudante de graduação do primeiro ano e utiliza vocabulário acessível, embora alguns termos como 'curvatura do espaço-tempo' e 'teorias como a gravidade quântica em loop' possam exigir conhecimento prévio. O texto é informativo, mas não faz uso de analogias criativas para facilitar o entendimento, o que limita o engajamento e o interesse do leitor. O parágrafo é claro, mas falta criatividade e elementos que tornem o conceito mais próximo do cotidiano do estudante.


## 4. Dicas de Consultoria (Na Trincheira)

1.  **Contexto dita a Temperatura:** Se estiver construindo um sistema de SQL-generation ou RAG corporativo, trave a `temperature` entre `0.0` e `0.1`. Alucinações caem drasticamente.
2.  **Monitoramento Contínuo:** Faça logs dos parâmetros junto com os inputs e outputs. Use frameworks como DeepEval em pipelines de CI/CD para evitar regressões nas atualizações de modelos.

## 5. BÔNUS NÍVEL AVANÇADO: Hyperparameter Optimization (HPO) de LLMs

Como plenos, sabemos que otimização na mão não é ideal. Abaixo, implementamos uma busca em grade (Grid Search) simplificada para encontrar o parâmetro de temperatura ideal que maximiza um score do DeepEval.

In [49]:
import numpy as np

conciseness_metric = GEval(
    name="Conciseness",
    criteria="A resposta vai direto ao ponto sem enrolação ou preâmbulos desnecessários.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.6
)

temperatures = [0.1, 0.5, 0.9]
best_score = -1
best_temp = None

print("--- Iniciando Otimização de Parâmetros ---")
for temp in temperatures:
    current_params = {"temperature": temp, "max_tokens": 200}
    out = generate_response("Resuma a teoria da relatividade.", current_params)

    tc = LLMTestCase(input="Resuma a teoria da relatividade.", actual_output=out)
    conciseness_metric.measure(tc)

    score = conciseness_metric.score
    print(f"Temp: {temp} -> Score de Concisão: {score}")
    print(f"Resposta Gerada:\n{out}\n")

    if score > best_score:
        best_score = score
        best_temp = temp

print(f"\n>>> Melhor Temperatura Encontrada para este Prompt: {best_temp} (Score: {best_score})")

--- Iniciando Otimização de Parâmetros ---


Output()

Temp: 0.1 -> Score de Concisão: 0.6737249324587599
Resposta Gerada:
A teoria da relatividade, proposta por Albert Einstein no início do século XX, é dividida em duas partes: a relatividade restrita e a relatividade geral.

1. **Relatividade Restrita (1905)**: Esta teoria introduz a ideia de que as leis da física são as mesmas para todos os observadores que estão em movimento uniforme entre si. Um dos postulados fundamentais é que a velocidade da luz no vácuo é constante e independente do movimento da fonte ou do observador. A relatividade restrita também trouxe a famosa equação \(E=mc^2\), que relaciona energia (E) e massa (m), mostrando que a massa pode ser convertida em energia e vice-versa. Essa teoria altera a compreensão do espaço e do tempo, que não são absolutos, mas sim interligados em um continuum chamado espaço-tempo.

2. **Relatividade Geral (1915)**: Esta teoria expande os conceitos da relatividade restrita



Output()

Temp: 0.5 -> Score de Concisão: 0.6942119992158216
Resposta Gerada:
A teoria da relatividade, desenvolvida por Albert Einstein no início do século XX, é dividida em duas partes: a relatividade restrita e a relatividade geral.

1. **Relatividade Restrita (1905)**: Esta teoria aborda a física em velocidades constantes e próximas à velocidade da luz. Seus principais postulados são:
   - As leis da física são as mesmas para todos os observadores que estão em movimento uniforme (ou seja, não estão acelerando).
   - A velocidade da luz no vácuo é constante e igual para todos os observadores, independentemente do movimento da fonte de luz ou do observador.

A relatividade restrita introduz conceitos como a dilatação do tempo (o tempo passa mais devagar para objetos em movimento rápido em relação a um observador em repouso) e a contração do comprimento (objetos em movimento rápido parecem mais curtos na direção do movimento). Também estabelece a famosa equação \(E=mc^2\),



Output()

Temp: 0.9 -> Score de Concisão: 0.6026149782894852
Resposta Gerada:
A teoria da relatividade, desenvolvida por Albert Einstein no início do século XX, compreende duas partes principais: a relatividade restrita e a relatividade geral.

1. **Relatividade Restrita (1905)**: Esta teoria introduz conceitos fundamentais sobre o espaço e o tempo, demonstrando que eles não são absolutos, mas sim interdependentes. A relatividade restrita postula que as leis da física são as mesmas para todos os observadores inerciais e que a velocidade da luz no vácuo é constante, independentemente do movimento da fonte ou do observador. Um dos resultados mais famosos dessa teoria é a famosa equação \(E=mc^2\), que relaciona energia (E) e massa (m), mostrando que a massa pode ser convertida em energia e vice-versa.

2. **Relatividade Geral (1915)**: Esta extensão da teoria da relatividade inclui a gravidade como uma propriedade do espaço-tempo. Einstein prop


>>> Melhor Temperatura Encontrada para este Prompt: